In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_TABLE = "gs_carbon.silver.openmeteo_weather"
TARGET_TABLE = "gs_carbon.gold.climate_risk"

In [0]:
df = spark.table(SOURCE_TABLE)

df = (
    df
    .withColumn(
        "temperature_score",
        F.when(
            F.col("temperature_mean") > 35,
            3
        )
        .when(
            F.col("temperature_mean") > 30,
            2
        )
        .when(
            F.col("temperature_mean") > 25,
            1
        )
        .otherwise(0)
    )
)

df = (
    df
    .withColumn(
        "precipitation_score",
        F.when(
            F.col("precipitation_sum") == 0,
            3
        )
        .when(
            F.col("precipitation_sum") < 5,
            2
        )
        .when(
            F.col("precipitation_sum") < 10,
            1
        )
        .otherwise(0)
    )
)

df = (
    df
    .withColumn(
        "wind_score",
        F.when(
            F.col("wind_speed_max") > 50,
            3
        )
        .when(
            F.col("wind_speed_max") > 30,
            2
        )
        .when(
            F.col("wind_speed_max") > 20,
            1
        )
        .otherwise(0)
    )
)

df = (
    df
    .withColumn(
        "risk_score",
        F.col("temperature_score")
        + F.col("precipitation_score")
        + F.col("wind_score")
    )
)

df = (
    df
    .withColumn(
        "DSC_CATEGORIA_RISCO",
        F.when(
            F.col("risk_score") >= 9,
            "Crítico"
        )
        .when(
            F.col("risk_score") >= 6,
            "Alto"
        )
        .when(
            F.col("risk_score") >= 3,
            "Moderado"
        )
        .otherwise(
            "Baixo"
        )
    )
)

gold_df = (
    df.select(
        "uf",
        "reference_date",
        "temperature_mean",
        "precipitation_sum",
        "wind_speed_max",
        "risk_score",
        "DSC_CATEGORIA_RISCO"
    )
)

spark.sql("""
CREATE SCHEMA IF NOT EXISTS gs_carbon.gold
""")

(
    gold_df.write
           .format("delta")
           .mode("overwrite")
           .option("overwriteSchema", "true")
           .saveAsTable(TARGET_TABLE)
)

print(
    f"Gold atualizada: {TARGET_TABLE}"
)

In [0]:
%sql
SELECT * FROM gs_carbon.gold.climate_risk